# Question 1

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import galsim
import galsim.hsm
from pathlib import Path
 
 
# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
DATA_DIR = Path("data") # Directory where input data is stored
OUTPUT_DIR = Path("outputs") # Directory where outputs will be saved
OUTPUT_DIR.mkdir(exist_ok=True) # Create output directory if it doesn't exist
 
STAMPS_FILE = DATA_DIR / "galaxy_stamps.fits" # FITS file containing galaxy stamps

## Question 1a

In [2]:
def _pixel_grids(image: galsim.Image):
    """
    Return (x, y) pixel-offset arrays relative to the image centre.
    Parameters
    ----------
    image : galsim.Image
        The image for which to compute the pixel grids.
    Returns
    -------
    x, y : np.ndarray
        2D arrays of shape (ny, nx) containing the x and y pixel offsets
        from the image centre.
    """

    arr = image.array # Get the image array
    ny, nx = arr.shape # Get the dimensions of the image
    cx, cy = (nx - 1) / 2.0, (ny - 1) / 2.0 # Compute the centre coordinates
    x_idx  = np.arange(nx, dtype=float) - cx # Create x pixel offsets
    y_idx  = np.arange(ny, dtype=float) - cy # Create y pixel offsets
    x, y   = np.meshgrid(x_idx, y_idx)   # Shape (ny, nx)
    return x, y

In [3]:
def unweighted_ellipticity(image: galsim.Image):
    """
    Estimate (e1, e2) from unweighted second-order moments.
 
        Q_ij = sum_{pq} I(p,q) * dx_i * dx_j
        e1 = (Q11 - Q22) / (Q11 + Q22 + 2 * sqrt(Q11 * Q22 - Q12^2))
        e2 = Q12 / (Q11 + Q22 + 2 * sqrt(Q11 * Q22 - Q12^2))
 
    Measurements with |e| > 1 (noise-dominated / diverging) are returned
    as (nan, nan) because they indicate a failed centroid or a stamp where
    the noise dominates the moment denominator.
 
    Returns
    -------
    (e1, e2) : floats, or (nan, nan) on failure.
    """
    arr   = image.array.astype(float) # Get the image array as float for calculations
    x, y  = _pixel_grids(image) # Get the pixel-offset relative to centre grids
 
    I_sum = arr.sum() # Total intensity (zeroth moment)
    if I_sum <= 0:
        return np.nan, np.nan # No signal, so return NaN for ellipticity
 
    # Centroid computations (first moments)
    xbar = (arr * x).sum() / I_sum # Centroid x-coordinate
    ybar = (arr * y).sum() / I_sum # Centroid y-coordinate
    dx   = x - xbar # Pixel offsets from centroid in x
    dy   = y - ybar # Pixel offsets from centroid in y
 
    # Second moments
    Q11   = (arr * dx * dx).sum() # Q11 = sum(I * dx^2)
    Q22   = (arr * dy * dy).sum() # Q22 = sum(I * dy^2)
    Q12   = (arr * dx * dy).sum() # Q12 = sum(I * dx * dy)
    denom = Q11 + Q22 + 2 * np.sqrt(Q11 * Q22 - Q12**2) # Denominator for ellipticity calculation
 
    if denom <= 0:
        return np.nan, np.nan # Denominator is zero or negative, likely noise-dominated, so return NaN
 
    e1 = (Q11 - Q22) / denom # Ellipticity component e1
    e2 = Q12 / denom # Ellipticity component e2
 
    # Physical bound: |e| <= 1 for any non-negative intensity profile.
    # Values outside this range signal a diverging / noise-dominated stamp.
    if abs(e1) > 1.0 or abs(e2) > 1.0:
        return np.nan, np.nan # Return NaN for unphysical ellipticity values
 
    return e1, e2

In [4]:
def _gaussian_weight(dx, dy, sigma):
    """
    Circular Gaussian W = exp(-(dx^2 + dy^2) / (2 sigma^2)).
    
    Parameters
    ----------
    dx, dy : np.ndarray
        Pixel offsets from the centroid in x and y directions.
    sigma : float
        Standard deviation of the Gaussian weight function in pixels.

    Returns
    -------
    W : np.ndarray
        2D array of the same shape as dx and dy containing the Gaussian weights.
    """

    return np.exp(-(dx**2 + dy**2) / (2.0 * sigma**2))
 
 
def weighted_ellipticity_raw(image: galsim.Image, fwhm_px: float = 4.0):
    """
    Estimate (e1, e2) from weighted second-order moments before
    responsivity correction.
 
    The weight function is a circular Gaussian with sigma = FWHM / (2*sqrt(2*ln2)).
    The centroid is taken from the unweighted first moments (single-pass).
 
    Returns
    -------
    (e1_raw, e2_raw) : floats, or (nan, nan) on failure.
    """
    arr  = image.array.astype(float) # Get the image array as float for calculations
    x, y = _pixel_grids(image) # Get the pixel-offset relative to centre grids
 
    sigma = fwhm_px / (2.0 * np.sqrt(2.0 * np.log(2.0))) # Convert FWHM to sigma for Gaussian weight
 
    # Unweighted centroid
    I_sum = arr.sum() # Total intensity (zeroth moment)
    if I_sum <= 0:
        return np.nan, np.nan # No signal, so return NaN for ellipticity
    xbar = (arr * x).sum() / I_sum # Centroid x-coordinate
    ybar = (arr * y).sum() / I_sum # Centroid y-coordinate
    dx   = x - xbar # Pixel offsets from the centroid in x direction
    dy   = y - ybar # Pixel offsets from the centroid in y direction
 
    # Gaussian weight centred on the unweighted centroid
    W = _gaussian_weight(dx, dy, sigma) # Compute the Gaussian weights for each pixel based on the offsets and sigma
    WI = W * arr # Weighted intensity for each pixel
    WI_sum = WI.sum() # Total weighted intensity (zeroth moment with weights)
    if WI_sum <= 0:
        return np.nan, np.nan # No signal in the weighted image, so return NaN for ellipticity
 
    # Weighted second moments
    Q11   = (WI * dx * dx).sum() # Q11 = sum(W * I * dx^2)
    Q22   = (WI * dy * dy).sum() # Q22 = sum(W * I * dy^2)
    Q12   = (WI * dx * dy).sum() # Q12 = sum(W * I * dx * dy)
    denom = Q11 + Q22 + 2 * np.sqrt(Q11 * Q22 - Q12**2) # Denominator for ellipticity calculation
 
    if denom <= 0:
        return np.nan, np.nan # Denominator is zero or negative, likely noise-dominated, so return NaN
 
    e1 = (Q11 - Q22) / denom # Ellipticity component e1
    e2 = Q12 / denom # Ellipticity component e2
    return e1, e2

In [5]:
def apply_responsivity_correction(e_raw: np.ndarray) -> np.ndarray:
    """
    Apply a shear responsivity correction to raw weighted ellipticities.
 
    The Gaussian weight function dilutes the shear signal because it rounds
    the effective galaxy profile. The measured ellipticity is related to
    the true shear by <e_raw> = 2 R gamma, where the responsivity is:
 
        R = 1 - sigma_e^2
 
    with sigma_e the per-component intrinsic ellipticity dispersion
    (Bernstein & Jarvis 2002; Kaiser 2000). We estimate sigma_e^2 from
    the ensemble variance of e_raw itself (iterating once is sufficient).
 
    Returns
    -------
    e_corrected : ndarray, shape (N, 2), with NaNs propagated.
    """
    valid = np.isfinite(e_raw[:, 0]) & np.isfinite(e_raw[:, 1]) # Identify valid measurements (finite values in both components)
    if valid.sum() < 2:
        return e_raw.copy() # Not enough valid measurements to estimate variance, so return original array
 
    # Per-component variance -> responsivity
    var_e1 = np.var(e_raw[valid, 0]) # Variance of e1 component from valid measurements
    var_e2 = np.var(e_raw[valid, 1]) # Variance of e2 component from valid measurements
    sigma_e_sq = 0.5 * (var_e1 + var_e2) # Average over both components
    R = 1.0 - sigma_e_sq # Responsivity factor
 
    print(f"Responsivity correction: sigma_e^2 = {sigma_e_sq:.4f},  R = {R:.4f}")
 
    e_corr = e_raw.copy() # Create a copy of the raw ellipticities to apply the correction
    e_corr[valid] /= R # Apply the responsivity correction to valid measurements
    return e_corr

In [6]:
def hsm_ellipticity(image: galsim.Image):
    """
    Estimate (e1, e2) using GalSim's HSM adaptive-moments algorithm
    (Hirata & Seljak 2003; Mandelbaum et al. 2005).
 
    HSM iteratively adapts an elliptical Gaussian weight to match the
    galaxy shape, giving an unbiased (at linear order) shear estimator
    without needing a separate responsivity correction.
 
    Returns
    -------
    (e1, e2) : floats, or (nan, nan) if HSM fails.
    """
    try:
        result = galsim.hsm.FindAdaptiveMom(image) # Use GalSim's HSM algorithm to find adaptive moments and ellipticity
        return result.observed_shape.g1, result.observed_shape.g2 # Return the ellipticity components from the HSM result
    except galsim.hsm.HSMError:
        return np.nan, np.nan # Return NaN if HSM fails (e.g., due to noise or convergence issues)

In [7]:
def measure_all_ellipticities(stamps_path: Path):
    """
    Load all galaxy stamps and compute (e1, e2) for each estimator.
 
    Returns
    -------
    dict with keys 'unweighted', 'weighted', 'hsm', each an ndarray
    of shape (N, 2) containing (e1, e2) per galaxy (NaN on failure).
    """
    galaxy_ims = galsim.fits.readMulti(str(stamps_path)) # Load galaxy stamps from the FITS file as a list of GalSim images
    n_gals = len(galaxy_ims) # Get the number of galaxy stamps loaded
    print(f"Loaded {n_gals} galaxy stamps from {stamps_path}")
 
    e_unw = np.full((n_gals, 2), np.nan) # Initialise array for unweighted ellipticities with NaNs
    e_wgt = np.full((n_gals, 2), np.nan) # Initialise array for weighted ellipticities with NaNs
    e_hsm = np.full((n_gals, 2), np.nan) # Initialise array for HSM ellipticities with NaNs
    
    # Compute ellipticities for each galaxy stamp using the three estimators
    for i, im in enumerate(galaxy_ims):
        e_unw[i] = unweighted_ellipticity(im) # Compute unweighted ellipticity for the current galaxy stamp
        e_wgt[i] = weighted_ellipticity_raw(im, fwhm_px=4.0) # Compute raw weighted ellipticity for the current galaxy stamp
        e_hsm[i] = hsm_ellipticity(im) # Compute HSM ellipticity for the current galaxy stamp
 
    # Apply responsivity correction to the weighted estimator
    print("Applying responsivity correction to weighted moments...")
    e_wgt = apply_responsivity_correction(e_wgt) # Apply the responsivity correction to the raw weighted ellipticities
    
    # Compile results into a dictionary for easy access
    results = {
        "unweighted": e_unw,
        "weighted":   e_wgt,
        "hsm":        e_hsm,
    }
 
    # Summary statistics (valid measurements only)
    print("\nSummary statistics (valid measurements only):")
    for name, arr in results.items():
        valid   = np.isfinite(arr[:, 0]) # Identify valid measurements based on finite values in the first component (e1)
        n_valid = valid.sum() # Count the number of valid measurements for this estimator
        print(
            f"  {name:12s}: {n_valid:6d}/{n_gals} valid | "
            f"<e1> = {arr[valid, 0].mean():+.4f}  σ(e1) = {arr[valid, 0].std():.4f} | "
            f"<e2> = {arr[valid, 1].mean():+.4f}  σ(e2) = {arr[valid, 1].std():.4f}"
        )
 
    return results

In [8]:
def plot_ellipticity_comparison(results: dict, output_dir: Path):
    """
    Three diagnostic figures:
      1. Scatter e1 vs e2 per method (physical ±1 window).
      2. Histograms of e1 and e2 per method.
      3. Direct scatter of each method vs HSM, with 1:1 line and Pearson r.
    """
    methods = ["unweighted", "weighted", "hsm"]
    colours = {"unweighted": "steelblue", "weighted": "tomato", "hsm": "seagreen"}
    labels  = {
        "unweighted": "Unweighted moments",
        "weighted": "Weighted moments (Gaussian, FWHM=4px)",
        "hsm": "HSM (Hirata-Seljak-Mandelbaum)",
    }
 
    # ---- 1. Scatter e1 vs e2 (physical window) ------------------------------
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True, sharex=True)
    for ax, method in zip(axes, methods):
        arr = results[method]
        valid = np.isfinite(arr[:, 0])
        ax.scatter(arr[valid, 0], arr[valid, 1], s=3, alpha=0.35, color=colours[method], rasterized=True)
        ax.axhline(0, lw=0.7, color="k", ls="--")
        ax.axvline(0, lw=0.7, color="k", ls="--")
        ax.set_xlim(-1, 1)
        ax.set_ylim(-1, 1)
        ax.set_xlabel(r"$e_1$", fontsize=13)
        ax.set_title(labels[method], fontsize=10)
        n_valid = valid.sum()
        ax.text(0.97, 0.97, f"N={n_valid}/{len(valid)}", transform=ax.transAxes, ha="right", va="top", fontsize=9)
    axes[0].set_ylabel(r"$e_2$", fontsize=13)
    fig.suptitle(r"Ellipticity estimates per method: $e_1$ vs $e_2$", fontsize=13)
    fig.tight_layout()
    out = output_dir / "q1_scatter_e1_e2.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {out}")
 
    # ---- 2. Histograms -------------------------------------------------------
    fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
    bins = np.linspace(-1, 1, 60)
    for col, method in enumerate(methods):
        arr = results[method]
        valid = np.isfinite(arr[:, 0])
        for row, comp in enumerate(["e1", "e2"]):
            ax = axes[row, col]
            data = arr[valid, row]
            # Clip to [-1,1] for histogram (unweighted can have corrected
            # values slightly outside if R<1 but these are valid)
            data_clipped = np.clip(data, -1, 1)
            ax.hist(data_clipped, bins=bins, color=colours[method], alpha=0.7, density=True)
            ax.axvline(data.mean(), color="k", lw=1.5, ls="--", label=f"mean = {data.mean():.4f}")
            ax.set_title(f"{labels[method]}\n" f"${comp}$   σ = {data.std():.4f}", fontsize=9,
            )
            ax.legend(fontsize=8)
            if row == 1:
                ax.set_xlabel(f"${comp}$", fontsize=12)
            if col == 0:
                ax.set_ylabel("Density", fontsize=11)
    fig.suptitle("Ellipticity distributions per method", fontsize=13)
    fig.tight_layout()
    out = output_dir / "q1_histograms.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {out}")
 
    # ---- 3. Method vs HSM scatter -------------------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(11, 10))
    e_hsm = results["hsm"]
    hsm_valid = np.isfinite(e_hsm[:, 0])
 
    compare_pairs = [
        ("unweighted", 0, "e_1"),
        ("unweighted", 1, "e_2"),
        ("weighted",   0, "e_1"),
        ("weighted",   1, "e_2"),
    ]
    for ax, (method, idx, comp_label) in zip(axes.flat, compare_pairs):
        arr = results[method]
        both_valid = hsm_valid & np.isfinite(arr[:, 0])
        x_vals = e_hsm[both_valid, idx]
        y_vals = arr[both_valid, idx]
 
        ax.scatter(x_vals, y_vals, s=3, alpha=0.35, color=colours[method], rasterized=True)
        lim = 0.5
        ax.plot([-lim, lim], [-lim, lim], "k--", lw=0.9, label="1:1")
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_xlabel(f"HSM  ${comp_label}$", fontsize=12)
        ax.set_ylabel(f"{labels[method].split('(')[0].strip()}  ${comp_label}$", fontsize=11)
        ax.legend(fontsize=8)
 
        r = np.corrcoef(x_vals, y_vals)[0, 1]
        ax.text(0.04, 0.95, f"r = {r:.3f}", transform=ax.transAxes, fontsize=9, va="top")
 
    fig.suptitle("Method comparison vs HSM", fontsize=13)
    fig.tight_layout()
    out = output_dir / "q1_comparison_vs_hsm.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {out}")

In [9]:
def main():
    results = measure_all_ellipticities(STAMPS_FILE) # Measure ellipticities for all galaxy stamps using the defined estimators
    plot_ellipticity_comparison(results, OUTPUT_DIR) # Generate diagnostic plots comparing the ellipticity estimates from different methods
 
    # Save for use in Q2
    np.save(OUTPUT_DIR / "ellipticities_unweighted.npy", results["unweighted"]) # Save unweighted ellipticity results to a .npy file in the outputs directory
    np.save(OUTPUT_DIR / "ellipticities_weighted.npy", results["weighted"]) # Save weighted ellipticity results to a .npy file in the outputs directory
    np.save(OUTPUT_DIR / "ellipticities_hsm.npy", results["hsm"]) # Save HSM ellipticity results to a .npy file in the outputs directory
    print("\nEllipticity arrays saved to outputs/.")
    return results
 
 
if __name__ == "__main__":
    main()

Loaded 100000 galaxy stamps from data/galaxy_stamps.fits


/tmp/ipykernel_79101/3779625280.py:34: RuntimeWarning: invalid value encountered in sqrt
  denom = Q11 + Q22 + 2 * np.sqrt(Q11 * Q22 - Q12**2) # Denominator for ellipticity calculation
/tmp/ipykernel_79101/1695803961.py:58: RuntimeWarning: invalid value encountered in sqrt
  denom = Q11 + Q22 + 2 * np.sqrt(Q11 * Q22 - Q12**2) # Denominator for ellipticity calculation


Applying responsivity correction to weighted moments...
Responsivity correction: sigma_e^2 = 0.0001,  R = 0.9999

Summary statistics (valid measurements only):
  unweighted  :  92351/100000 valid | <e1> = -0.0002  σ(e1) = 0.1047 | <e2> = +0.0001  σ(e2) = 0.0676
  weighted    :  99996/100000 valid | <e1> = +0.0000  σ(e1) = 0.0132 | <e2> = -0.0000  σ(e2) = 0.0064
  hsm         : 100000/100000 valid | <e1> = +0.0001  σ(e1) = 0.0572 | <e2> = +0.0001  σ(e2) = 0.0573
Saved outputs/q1_scatter_e1_e2.png
Saved outputs/q1_histograms.png
Saved outputs/q1_comparison_vs_hsm.png

Ellipticity arrays saved to outputs/.
